# Script Outline



## Prepare Workspace

#### Import Packages

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs

#### File paths

In [ ]:
# Define user
user = os.getlogin()

# Working directories
path_sp  = os.path.join('C:\\Users', 'jfontes', 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_git = os.path.join('C:\\Users', 'jfontes', 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

# Set file paths
path_config = os.path.join(path_git, 'Pipeline', 'Python Code', 'Census', 'aa_config')
path_out    = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')

#### User Defined Functions/Objects

In [ ]:
## User defined functions
exec(open(os.path.join(path_config, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

## Prepare Inputs for Importing ACS Data

#### Import ACS tables/variables mapping and FIPS mapping

In [ ]:
## Import Variable Mapping
df_vars   = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'ACS')
df_inputs = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'Inputs'
                        , dtype = {'msa': object})

# Organize inputs into run
indicator_name = df_inputs['indicator_name'].values[0]
year_start = int(df_inputs['year_start'].values[0])
year_end   = int(df_inputs['year_end'  ].values[0])
sp_folder_out = df_inputs['sp_folder'].values[0]

# Subset variables
df_vars = df_vars[df_vars['Indicator Name'] == indicator_name]
df_vars = df_vars[df_vars['Include'] == 'Yes']

# Set tables and variables to import
list_vars = ['NAME'] + df_vars['ID'].to_list()
tables = df_vars['Table'].unique()

# Set years
years_to_import = list(range(year_start, year_end+1))
years_to_import.remove(2020)


## Import County FIPS mapping
df_fips = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx')
                        , sheet_name = 'FIPSmapping'
                        , dtype = {'State FIPS': object, 'County FIPS': object})

# Convert to dictionary
dict_fips = df_fips[
                (df_fips['State'].isin(df_inputs['states'].values)) 
                & (df_fips['County Name'].isin(df_inputs['counties'].values))
]
dict_fips = dict_fips[['State FIPS', 'County FIPS']]

dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()

for key in list(dict_fips.keys()):
    dict_fips[key] = ",".join(dict_fips[key])

    
# Set MSA
df_inputs['msa'] = df_inputs['msa'].astype("string")
msa_to_import = df_inputs['msa'].dropna().values
msa_to_import = ",".join(msa_to_import)



# view
print(dict_fips)
print(msa_to_import)
print(tables)
print(indicator_name)
print(year_start)
print(year_end)
df_vars.head()

## Import Data

#### Create Census Tracts Table

In [ ]:

# initialize empty list to store data frames
list_df_acs = []

# only want one table
df_table = df_vars[df_vars['Table'] == tables[0]]
list_table_vars = ['NAME'] + df_table['ID'].to_list()
variables = ",".join(list_table_vars)


# pull data, subset to just ID fields
for state in list(dict_fips.keys()):
    for year in tqdm(years_to_import):
        try:
            temp = acs1_state_county(api_Key     = api_key
                                     , variables = variables
                                     , year      = year
                                     , state     = state
                                     , county    = dict_fips[state])
            
            temp = temp[['NAME', 'state', 'county', 'Year']]
            list_df_acs.append(temp)
            
        except:
            pass

# combine all years and counties
df_acs_raw = pd.concat(list_df_acs)

# merge county name onto table
df_acs_raw = df_acs_raw.merge(df_fips[['County FIPS', 'County Name']], left_on = 'county', right_on = 'County FIPS')
df_acs_raw.drop(['County FIPS'], axis = 1, inplace = True)
df_acs_raw.head()

#### Import ACS Data by Census Tracts

In [ ]:

# initialize empty list to store data frames
# import multiple years and counties
list_df_acs = []

# iterate through each table (pulling all tables at once fails because the URL is too long - i think)
for table in tables:

    # keep track of tables being imported
    print(table)
    list_df_tables = []

    # subset to variables within looped table
    df_table = df_vars[df_vars['Table'] == table]
    list_table_vars = ['NAME'] + df_table['ID'].to_list()
    variables = ",".join(list_table_vars)

    # pull all years and counties for each table
    for state in list(dict_fips.keys()):
        for year in tqdm(years_to_import):
            try:
                list_df_tables.append(
                    acs1_state_county(api_Key     = api_key
                                      , variables = variables
                                      , year      = year
                                      , state     = state
                                      , county    = dict_fips[state])
                )
            except:
                pass

    # combine all years and counties
    df_temp = pd.concat(list_df_tables)

    # left join data onto key
    df_acs_raw = df_acs_raw.merge(df_temp, on = ['NAME', 'state', 'county', 'Year'], how = 'left')

In [ ]:
# Check years and counties
print(df_acs_raw['Year'].unique())
# assert df_acs_raw['Year'  ].unique().tolist() == years_to_import


# view raw data
pd.set_option('display.max_columns', None)
print(df_acs_raw.shape)
print(df_acs_raw.Year.unique())
df_acs_raw.head(3)

## Data Cleaning

In [ ]:
# make copy
df_acs = df_acs_raw.copy()


# Replace "null" with 0
df_acs = df_acs.replace('null', np.nan)
df_acs = df_acs.dropna(axis = 0, how = "any")


# Melt data from wide to long
df_acs = pd.melt(df_acs
                  , id_vars = ['County Name', 'NAME', 'state', 'county', 'Year'] 
                  , var_name = 'ID'
                  , value_name = 'Population'
                 )


# Convert imported values to numeric
df_acs['Population'] = df_acs['Population'].apply(pd.to_numeric)


# Merge label 2
df_acs = df_acs.merge(df_vars[['ID', 'Table Name', 'Label', 'Variable', 'Race_Ethnicity']], on = 'ID', how = 'left')


# Subset
df_acs = df_acs[['ID', 'Table Name', 'Label', 'County Name', 'NAME', 'state', 
                   'county', 'Year', 'Variable', 'Race_Ethnicity', 'Population']]


# Manually check column names and clean as needed
df_acs = df_acs.rename(columns = {
    'ID':'Table ID'
    , 'state':'State FIPS'
    , 'county':'County FIPS'
})


# view
df_acs.head()

## Organize Exports

In [ ]:
# Sort by census tract then by year then by race/ethnicity
df_acs['Race_Ethnicity_sort'] = pd.Categorical(df_acs['Race_Ethnicity'], ['All'
                                                                 , 'AMERICAN INDIAN AND ALASKA NATIVE ALONE'
                                                                 , 'ASIAN ALONE'
                                                                 , 'BLACK OR FRICAN AMERICAN ALONE'
                                                                 , 'HISPANIC OR LATINO'
                                                                 , 'NATIVE HAWAIIAN AND OTHER PACIFIC ISLANDER ALONE'
                                                                 , 'WHITE ALONE'
                                                                 , 'WHITE ALONE, NOT HISPANIC OR LATINO'
                                                                 , 'SOME OTHER RACE ALONE', 'TWO OR MORE RACES'])


# sort and then remove categorical field
df_acs = df_acs.sort_values(by = ['NAME', 'Year', 'Race_Ethnicity_sort'], ascending = True)
df_acs = df_acs.drop(['Race_Ethnicity_sort'], axis = 1)

In [ ]:
df_msa

In [ ]:
# Create MPO and MSA groupings
df_mpo = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'MPO')
df_msa = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'MSA')


# Merge groupings
df_acs = df_acs.merge(df_mpo, on = ['County Name', 'State FIPS'], how = 'left')
df_acs = df_acs.merge(df_msa, on = ['County Name', 'State FIPS'], how = 'left')


# reorder columns
cols = ['Table ID', 'Table Name', 'Label', 'State FIPS', 'MPO', 'MSA', 'County Name', 
        'County FIPS', 'Year', 'Variable', 'Race_Ethnicity', 'Population']
df_acs = df_acs[cols]


# missing values represent a population of 0
df_acs['Population'] = df_acs['Population'].fillna(0)


# view
df_acs.head()

In [ ]:
## Groupings roll up
df_acs1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 'Tract ID', 'NAME', 
                          'Variable', 'Year', 'Race_Ethnicity'
                         ], as_index = False, sort = False)['Population'].sum()
df_acs1['props'] = df_acs1['Population'] / df_acs1[df_acs1['Variable'] != 'Total'].groupby(['NAME', 'Year'])['Population'].transform('sum')


# Counties
df_counties1 = df_acs.groupby(['State FIPS', 'County FIPS', 'County Name', 
                               'Variable', 'Year', 'Race_Ethnicity'
                              ], as_index = False, sort = False)['Population'].sum()
df_counties1['props'] = df_counties1['Population'] / df_counties1[df_counties1['Variable'] != 'Total'].groupby(['State FIPS', 'County FIPS', 'Year'])['Population'].transform('sum')


# MSA
df_msa1 = df_acs.groupby(['State FIPS', 'MSA', 
                          'Variable','Year', 'Race_Ethnicity'
                         ], as_index = False, sort = False)['Population'].sum()
df_msa1['props'] = df_msa1['Population'] / df_msa1[df_msa1['Variable'] != 'Total'].groupby(['MSA', 'Year'])['Population'].transform('sum')


# MPO
df_mpo1 = df_acs.groupby(['State FIPS', 'MPO', 
                          'Variable', 'Year', 'Race_Ethnicity'
                         ], as_index = False, sort = False)['Population'].sum()
df_mpo1['props'] = df_mpo1['Population'] / df_mpo1[df_mpo1['Variable'] != 'Total'].groupby(['MPO', 'Year'])['Population'].transform('sum')



# missing values represent a population of 0
df_acs1     ['props'] = df_acs1     ['props'].fillna(1)
df_counties1['props'] = df_counties1['props'].fillna(1)
df_msa1     ['props'] = df_msa1     ['props'].fillna(1)
df_mpo1     ['props'] = df_mpo1     ['props'].fillna(1)




# view
# df_acs1     .head()
# df_counties1.head()
# df_msa1     .head()
df_mpo1     .head()

In [ ]:
## Dcasts

# Tracts
df_acs2_pop = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS', 'Tract ID'
                                       , 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Population').reset_index()
df_acs2_prop = df_acs1.pivot_table(index = ['State FIPS', 'County FIPS', 'Tract ID'
                                       , 'County Name', 'NAME', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'props').reset_index()

# Counties
df_counties2_pop = df_counties1.pivot_table(index = ['State FIPS', 'County FIPS',
                                        'County Name', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Population').reset_index()
df_counties2_prop = df_counties1.pivot_table(index = ['State FIPS', 'County FIPS',
                                        'County Name', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'props').reset_index()

# MSA
df_msa2_pop = df_msa1.pivot_table(index = ['State FIPS', 'MSA', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Population').reset_index()
df_msa2_prop = df_msa1.pivot_table(index = ['State FIPS', 'MSA', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'props').reset_index()

# MPO
df_mpo2_pop = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'Population').reset_index()
df_mpo2_prop = df_mpo1.pivot_table(index = ['State FIPS', 'MPO', 'Year', 'Race_Ethnicity']
                                   , columns = 'Variable'
                                   , values = 'props').reset_index()


# missing values represent a population of 0
df_acs2_pop  = df_acs2_pop .fillna(0)
df_acs2_pop  = df_acs2_pop .fillna(0)
df_msa2_pop  = df_msa2_pop .fillna(0)
df_mpo2_pop  = df_mpo2_pop .fillna(0)
df_acs2_prop = df_acs2_prop.fillna(0)
df_acs2_prop = df_acs2_prop.fillna(0)
df_msa2_prop = df_msa2_prop.fillna(0)
df_mpo2_prop = df_mpo2_prop.fillna(0)


# view
df_msa2_prop.head(3)

#### Export

In [ ]:
# Set output name

name_output_long = ['ACS1 ', indicator_name, ' Long.xlsx']
name_output_wide = ['ACS1 ', indicator_name, ' Wide.xlsx']

name_output_long = "".join(name_output_long)
name_output_wide = "".join(name_output_wide)


In [ ]:
# # Export long
# with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_long), engine='xlsxwriter') as writer:
#     df_acs      .to_excel(writer, index = False, sheet_name = 'Full Tracts')
#     df_acs1     .to_excel(writer, index = False, sheet_name = 'Tracts'  )
#     df_counties1.to_excel(writer, index = False, sheet_name = 'Counties')
#     df_msa1     .to_excel(writer, index = False, sheet_name = 'MSA'     )
#     df_mpo1     .to_excel(writer, index = False, sheet_name = 'MPO'     )

In [ ]:
# # Export wide
# with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_wide), engine='xlsxwriter') as writer:
#     df_acs2_pop      .to_excel(writer, index = False, sheet_name = 'Tracts pop'   )
#     df_counties2_pop .to_excel(writer, index = False, sheet_name = 'Counties pop' )
#     df_msa2_pop      .to_excel(writer, index = False, sheet_name = 'MSA pop'      )
#     df_mpo2_pop      .to_excel(writer, index = False, sheet_name = 'MPO pop'      )
#     df_acs2_prop     .to_excel(writer, index = False, sheet_name = 'Tracts prop'  )
#     df_counties2_prop.to_excel(writer, index = False, sheet_name = 'Counties prop')
#     df_msa2_prop     .to_excel(writer, index = False, sheet_name = 'MSA prop'     )
#     df_mpo2_prop     .to_excel(writer, index = False, sheet_name = 'MPO prop'     )

In [ ]:
# https://api.census.gov/data/2015/acs/acs5?get=NAME,B00001_001E&for=metropolitan%20statistical%20area/micropolitan%20statistical%20area:10420&key=YOUR_KEY_GOES_HERE
def acs5_msa(api_Key, variables, year, msa): # need to include Race/Ethnicity argument for variables
    
    '''
    User defined function to import ACS 5 year estimates at the tract level
    Fixed inputs: [host_, dataset_, g_] to construct URL
    User inputs: [api_key_, variables_, year_, location_] to tell ACS that we have access with the API key and
                    to tell ACS which variables we want to import, what year, and which state and county
                    (currently can only do 1 state and county at a time)
    '''
    
    # Fixed inputs
    host_ = 'https://api.census.gov/data'
    dataset_ = '/acs/acs5'
    g_ = '?get='
    
    # User inputs
    api_key_ = f"&key={api_key}"
    variables_ = variables
    year_ = '/' + str(year)
    location_ = '&for=metropolitan%20statistical%20area/micropolitan%20statistical%20area:' + str(msa)
    
    # create url query
    query = f"{host_}{year_}{dataset_}{g_}{variables_}{location_}{api_key_}"
    
    # use requests package to call out to the API
    response = requests.get(query).text
    response = response.replace('null', '"null"')
    response = ast.literal_eval(response)
    
    # convert parsed response text to pandas df
    df_acs = pd.DataFrame(response[1:], columns = response[0])
    
    # apply year tag (probably need to apply a "Race_Ethnicity" tag too)
    df_acs['Year'] = year
    
    return df_acs

In [ ]:
msa_to_import = df_inputs['msa'].dropna().values
msa_to_import

In [ ]:
list_df_acs = []
error_log = []
# only want one table
df_table = df_vars[df_vars['Table'] == tables[0]]
list_table_vars = ['NAME'] + df_table['ID'].to_list()
variables = ",".join(list_table_vars)
year = 2021

for msa in msa_to_import:

    try:
        list_df_acs.append(
            acs5_msa(api_Key = api_key
                        , variables = variables
                        , year      = year
                        , msa       = msa)
        )
    except:
        error_log.append(msa)

error_log = list(set(error_log))
df_msa = pd.concat(list_df_acs)

In [ ]:
# only want one table
df_table = df_vars[df_vars['Table'] == tables[0]]
list_table_vars = ['NAME'] + df_table['ID'].to_list()
variables = ",".join(list_table_vars)
variables

year = 2022

In [ ]:
msa_to_import[0]

In [ ]:
temp = acs5_msa(api_Key         = api_key
                    , variables = variables
                    , year      = year
                    , msa       = msa_to_import[0])

temp.head()